[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alm-82/telecom-networks-weekly-projects/blob/main/week8/ChiFraud.ipynb)

In [ ]:
import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from transformers import BertTokenizerFast, BertForSequenceClassification, get_linear_schedule_with_warmup

In [18]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [19]:
TRAIN_PATH = r"C:/Users/fatem/Downloads/ChiFraud_train.csv"
TEST_PATH  = r"C:/Users/fatem/Downloads/ChiFraud_t2023.csv"
SEP = "\t"
TEXT_COL = "Text"
LABEL_COL = "Label_id"

In [20]:
MODEL_NAME = "bert-base-chinese"
MAX_LEN = 96
BATCH_SIZE = 8
EPOCHS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
CLIP_NORM = 1.0
GRAD_ACCUM_STEPS = 2  

In [21]:
train_df = pd.read_csv(TRAIN_PATH, sep=SEP, encoding="utf-8")
test_df  = pd.read_csv(TEST_PATH,  sep=SEP, encoding="utf-8")

train_df[TEXT_COL] = train_df[TEXT_COL].astype(str)
train_df[LABEL_COL] = train_df[LABEL_COL].astype(int)

test_df[TEXT_COL] = test_df[TEXT_COL].astype(str)
test_df[LABEL_COL] = test_df[LABEL_COL].astype(int)

NUM_LABELS = train_df[LABEL_COL].nunique()  
label_set = sorted(train_df[LABEL_COL].unique().tolist())

print("NUM_LABELS:", NUM_LABELS, "labels:", label_set)
print("Train shape:", train_df.shape, "Test shape:", test_df.shape)
print("Train label distribution:\n", train_df[LABEL_COL].value_counts().sort_index())


NUM_LABELS: 10 labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Train shape: (192267, 2) Test shape: (115553, 2)
Train label distribution:
 Label_id
0    166790
1      3607
2     11566
3       530
4       947
5      1607
6      1486
7      4392
8       483
9       859
Name: count, dtype: int64


In [22]:
tr_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df[LABEL_COL]
)

print("\nSplit:")
print("Train:", tr_df.shape, "Val:", val_df.shape)


Split:
Train: (173040, 2) Val: (19227, 2)


In [23]:
def stratified_cap(df, label_col, cap_per_class, seed=42):
    parts = []
    for lab, grp in df.groupby(label_col):
        if len(grp) > cap_per_class:
            grp = grp.sample(n=cap_per_class, random_state=seed)
        parts.append(grp)
    out = pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

CAP_TRAIN_PER_CLASS = 3000
CAP_VAL_PER_CLASS   = 600

tr_small = stratified_cap(tr_df, LABEL_COL, CAP_TRAIN_PER_CLASS, seed=42)
val_small = stratified_cap(val_df, LABEL_COL, CAP_VAL_PER_CLASS, seed=42)

print("\nUsing subset sizes:")
print("Train subset:", tr_small.shape)
print("Val subset  :", val_small.shape)
print("Train subset label dist:\n", tr_small[LABEL_COL].value_counts().sort_index())



Using subset sizes:
Train subset: (17321, 2)
Val subset  : (2591, 2)
Train subset label dist:
 Label_id
0    3000
1    3000
2    3000
3     477
4     852
5    1446
6    1338
7    3000
8     435
9     773
Name: count, dtype: int64


In [24]:
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

class FraudDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.texts = df[TEXT_COL].tolist()
        self.labels = df[LABEL_COL].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = FraudDataset(tr_small)
val_ds   = FraudDataset(val_small)
test_ds  = FraudDataset(test_df)  

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("\nBatches:", len(train_loader), len(val_loader))



Batches: 2166 324


In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
).to(device)



Using device: cpu


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [26]:
y_train = tr_small[LABEL_COL].values
classes = np.array(sorted(tr_small[LABEL_COL].unique()))
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)

full_w = np.ones(NUM_LABELS, dtype=np.float32)
for c, w in zip(classes, weights):
    full_w[int(c)] = w

class_weights = torch.tensor(full_w, dtype=torch.float).to(device)
loss_fn = CrossEntropyLoss(weight=class_weights)

print("\nClass weights:", class_weights.detach().cpu().numpy())



Class weights: [0.57736665 0.57736665 0.57736665 3.6312368  2.0329812  1.1978562
 1.2945441  0.57736665 3.9818392  2.2407503 ]


In [27]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)


In [28]:
def train_one_epoch(epoch):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    loop = tqdm(train_loader, desc=f"Train Epoch {epoch}")
    for step, batch in enumerate(loop, start=1):
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attn_mask)
        logits = outputs.logits

        loss = loss_fn(logits, labels)
        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()

        if step % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * GRAD_ACCUM_STEPS
        loop.set_postfix(loss=f"{(total_loss/step):.4f}")

    return total_loss / len(train_loader)


@torch.no_grad()
def evaluate(loader, title="Val"):
    model.eval()
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc=f"Eval {title}"):
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attn_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted")

    print(f"\n{title} Accuracy   : {acc:.4f}")
    print(f"{title} Macro F1   : {macro_f1:.4f}")
    print(f"{title} Weighted F1: {weighted_f1:.4f}")
    print("\nClassification report:")
    print(classification_report(all_labels, all_preds, digits=4))
    print("Confusion matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return macro_f1


best_macro = -1.0
save_dir = "chifraud_best"
os.makedirs(save_dir, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    avg_loss = train_one_epoch(epoch)
    print(f"\nEpoch {epoch} avg train loss: {avg_loss:.4f}")

    macro = evaluate(val_loader, title="Val")

    if macro > best_macro:
        best_macro = macro
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"\n✅ Saved best model to: {save_dir} (best macro-F1={best_macro:.4f})")




Train Epoch 1: 100%|████████████████████████████████████████████████| 2166/2166 [3:23:18<00:00,  5.63s/it, loss=0.5161]



Epoch 1 avg train loss: 0.5161


Eval Val: 100%|██████████████████████████████████████████████████████████████████████| 324/324 [05:30<00:00,  1.02s/it]



Val Accuracy   : 0.9649
Val Macro F1   : 0.9410
Val Weighted F1: 0.9652

Classification report:
              precision    recall  f1-score   support

           0     0.9671    0.9783    0.9727       600
           1     0.9671    0.9778    0.9725       361
           2     0.9865    0.9717    0.9790       600
           3     0.7742    0.9057    0.8348        53
           4     0.8990    0.9368    0.9175        95
           5     0.9349    0.9814    0.9576       161
           6     0.9650    0.9324    0.9485       148
           7     0.9929    0.9590    0.9757       439
           8     0.9167    0.9167    0.9167        48
           9     0.9518    0.9186    0.9349        86

    accuracy                         0.9649      2591
   macro avg     0.9355    0.9478    0.9410      2591
weighted avg     0.9659    0.9649    0.9652      2591

Confusion matrix:
[[587   5   2   0   1   0   2   1   0   2]
 [  3 353   1   2   1   0   0   0   0   1]
 [  6   1 583   1   0   9   0   0   0   

Train Epoch 2: 100%|████████████████████████████████████████████████| 2166/2166 [3:24:33<00:00,  5.67s/it, loss=0.1862]



Epoch 2 avg train loss: 0.1862


Eval Val: 100%|██████████████████████████████████████████████████████████████████████| 324/324 [05:21<00:00,  1.01it/s]



Val Accuracy   : 0.9676
Val Macro F1   : 0.9472
Val Weighted F1: 0.9675

Classification report:
              precision    recall  f1-score   support

           0     0.9686    0.9783    0.9735       600
           1     0.9698    0.9778    0.9738       361
           2     0.9865    0.9733    0.9799       600
           3     0.8627    0.8302    0.8462        53
           4     0.9667    0.9158    0.9405        95
           5     0.9349    0.9814    0.9576       161
           6     0.9514    0.9257    0.9384       148
           7     0.9839    0.9749    0.9794       439
           8     0.8889    1.0000    0.9412        48
           9     0.9419    0.9419    0.9419        86

    accuracy                         0.9676      2591
   macro avg     0.9455    0.9499    0.9472      2591
weighted avg     0.9678    0.9676    0.9675      2591

Confusion matrix:
[[587   5   2   0   0   0   3   1   0   2]
 [  4 353   1   1   0   0   0   0   0   2]
 [  5   1 584   1   0   9   0   0   0   

In [32]:
@torch.no_grad()
def predict_text(text: str):
    model.eval()
    enc = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN)
    enc = {k: v.to(device) for k, v in enc.items()}
    out = model(**enc)
    return torch.argmax(out.logits, dim=1).item()

print("\nManual tests:")
samples = [
    "尊敬的用户，您的银行卡异常，请立即点击此链接验证。",
    "紧急通知：您的账户已被冻结，请尽快处理。",
    "您好，明天一起吃饭吗？",
    "您已中奖，请提供身份证号领取奖金。",
     "梦之城重庆时时彩【赌场微信51732166】",                         
    "办理毕业证办文凭【微信:95400813168】本地制作发图确认",            
    "高价收个人银行卡四件套【Q：584929575】长期收购诚信合作",         
    "北京代还信用卡【微信同号：24600334005】刷卡本地真实商户",         
    "南京开医院诊断证明【Q/薇:274536493】支持送货上门",               
    "找妹子上门服务【V芯:42723906】一条龙全套过夜多少钱",           
    "一手网贷客户联系方式【Q:2838992396】实时数据保您满意",            
    "麻醉药哪能买到【微信:865495862】正品保证全天接单",
]
for s in samples:
    print(s, "=>", predict_text(s))


Manual tests:
尊敬的用户，您的银行卡异常，请立即点击此链接验证。 => 0
紧急通知：您的账户已被冻结，请尽快处理。 => 0
您好，明天一起吃饭吗？ => 0
您已中奖，请提供身份证号领取奖金。 => 0
梦之城重庆时时彩【赌场微信51732166】 => 1
办理毕业证办文凭【微信:95400813168】本地制作发图确认 => 3
高价收个人银行卡四件套【Q：584929575】长期收购诚信合作 => 4
北京代还信用卡【微信同号：24600334005】刷卡本地真实商户 => 6
南京开医院诊断证明【Q/薇:274536493】支持送货上门 => 7
找妹子上门服务【V芯:42723906】一条龙全套过夜多少钱 => 2
一手网贷客户联系方式【Q:2838992396】实时数据保您满意 => 9
麻醉药哪能买到【微信:865495862】正品保证全天接单 => 5


In [30]:
for lab in range(NUM_LABELS):
    print("\nLabel", lab)
    print(tr_small[tr_small[LABEL_COL]==lab][TEXT_COL].head(3).to_list())



Label 0
['花呗类似于信用卡的使用，不能取现，可以线上消费。我们在网购商品的时候，先使用花呗付款，到还款日之后还款就可以了，不会产生任何利息。', '“不一定要吃完，都尝尝味儿嘛，难得和柳副书记一起来街边店吃顿饭，不尝尝味儿怎没系那个呢，下次可就不一定有这个机会了，我给你介绍介绍，这个菜叫西府拼盘，这是很出名的凉菜，这个菜是砂锅牛蹄，配了十多种药材，口感肥而不厌，爽滑，味浓味道稍微有些辣……”赵德三将六道菜挨个向柳雪梅介绍了一番。', '*该信息由天眼查数据库分析得出，仅供参考，详情见页尾服务协议。']

Label 1
['lv/路易威登新年?小牛【钥匙扣】M75371牛牛钥匙扣包饰挂件小牛皮与标志性Monogram帆布【哇】特色水貂毛与饰钉配饰增添一份华贵质感【哇】为即将到来的5304牛年?埋下伏笔传递新的一年无尽的想象', '梦之城重庆时时彩【赌场微信51732166】', '提供今日立昂微(074629)行情数据，包括价格，各周期走势图，基本资料及实时新闻资讯，财务分析，公司介绍，分红派息信息，您还可使用富途牛牛开户交易苹果股票，为投资者提供参考决策数据。']

Label 2
['〖V芯:42723906雅静〗【找妹子美女酒店大保健一晚上全套过夜多少钱」〖V芯:21062769雅静〗〖哪里有找妹子上门服务联系方式」〖V芯:95966121雅静〗找哪里有找大保健SPA真实服务快餐全套〖V芯:97904262雅静〗【学院附近哪里有(美女)足疗洗浴服务」〖V芯:51705377雅静〗【妹子个人联系电话」〖V芯:61957723雅静〗找美女上门保健按摩服务〖V芯:26077200雅静〗车站附近妹子找服务〖V芯:39734971雅静〗一条龙全套莞式服务」〖V芯:92209294雅静〗\u3000\u3000（五）加强大数据支撑。加快全民健康信息平台建设，加强疫情防控相关数据的收集、汇聚和共享使用。加大“粤康码”、入境人员健康管理“一码通”等的推广使用力度，及时将检测结果、涉疫人员等信息共享到数据库。充分利用广东疫情防控大数据综合实战等平台，加强与国家、外省之间疫情防控数据实时共享更新，推进人员安全有序流动。定期进行大数据关联分析、风险研判，精准推送各类疫情防控信息。', '【微ａｋ2403ｋｋ】珠海斗门哪里有妹子按摩多少钱一晚b51yp', '〖V芯:34